## Libraries

In [1]:
import pandas as pd
import numpy as np

from collections import Counter

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn import clone
from sklearn.feature_selection import RFE
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, median_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid


## Import

In [109]:
X = pd.read_csv('../new_datasets/X_trainval_preprocessed.csv')
y = pd.read_csv('../new_datasets/y_trainval.csv').values.ravel()

C:\Users\rafad\AppData\Local\Temp\ipykernel_7796\1709477896.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  X = pd.read_csv('../new_datasets/X_trainval_preprocessed.csv')


In [110]:
X.head()

,Brand,model,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage,mileage_per_year,tax_engineSize,age_mileage,mpg_engineSize,miles_mpg,age
0,VW,Golf,Semi-Auto,28421.0,Petrol,120-240,11.417268,2.0,4.000000,0.0,3157.888889,NaN,255789.0,22.834536,324490.166831,9.0
1,Toyota,Yaris,Manual,4589.0,Petrol,120-240,47.900000,1.5,1.000000,0.0,764.833333,217.5,27534.0,71.850000,219813.100000,6.0
2,Audi,Q2,Semi-Auto,3624.0,Petrol,120-240,40.900000,1.5,4.000000,0.0,604.000000,217.5,21744.0,61.350000,148221.600000,6.0
3,Ford,Fiesta,Manual,9102.0,Petrol,120-240,65.700000,1.0,2.340306,0.0,1300.285714,145.0,63714.0,65.700000,598001.400000,7.0
4,BMW,2 Series,Manual,1000.0,Petrol,120-240,42.800000,1.5,3.000000,0.0,166.666667,217.5,6000.0,64.200000,42800.000000,6.0


In [4]:
y

array([22290, 13790, 24990, ...,  8399, 12990, 10495])

## Functions

In [78]:
def select_rfe(X, y, names, n_features=10):

    '''
    Selects the most revelant features using RFE with Ridge algorithm (wrapper method)

    The function fits an RFE selector on the data and returns the names of the selected features

    Args:
        X (pandas.DataFrame): Features of the training data
        y (array): Targets of X
        names (list or str): List containing the features in X (same order)
        n_features (int): Number of features to select (50 by default), In case of X having less features than the value, all available features are considered

    Returns
        selected_features (list): List with the names of the features selected by the RFE
    
    '''
    #In case our data has less than n_features (50 by default) we adjust the number of features
    n_features = min(n_features, X.shape[1])
    
    rfe = RFE(estimator=Ridge(alpha=1.0), n_features_to_select=n_features, step=1)
    rfe.fit(X, y)
    
    # PROTEÇÃO CRÍTICA: Iteramos apenas até ao limite seguro
    limit = min(len(names), len(rfe.support_))
    
    return [names[i] for i in range(limit) if rfe.support_[i]]

In [131]:
def select_ridge(X, y, names, alpha=5):

    """
    Selects relevant features based on Ridge regression coefficients (L2 regularization).

    The function fits a Ridge regression model and returns the names of
    features whose absolute coefficients are above a small threshold (since the coefficients never get to 0).


    Args:
    
        X (pandas.DataFrame): Feature of the training data

        y (array): Target of X.

        names (list or str): List containing the features in X (same order)

        alpha (float): Regularization strength for Ridge regression. Higher values increase coefficient shrinkage.

    Returns
        selected_features (list of str): Names of features whose absolute (Ridge) coefficient is greater than 1e-5.
        
    """

    ridge = Ridge(alpha=alpha)
    ridge.fit(X, y)
    
    # PROTEÇÃO CRÍTICA: O limite é o menor entre nomes e coeficientes
    limit = min(len(names), len(ridge.coef_))
    
    return [names[i] for i in range(limit) if abs(ridge.coef_[i]) > 1e-4]

In [ ]:
def select_lasso(X, y, names, alpha=12):

    """
    Selects relevant features using Lasso regression coefficients (L1 regularization).

    The function fits a Lasso regression model and returns the names of
    features whose coefficients are non-zero.

    Args:
    
        X (pandas.DataFrame): Feature of the training data

        y (array): Target of X.

        names (list or str): List containing the features in X (same order)

        alpha (float): Regularization strength for Lasso regression. Higher value increase sparsity and result in fewer selected features

    Returns
    
        selected_features (list of str): Names of features with non-zero Lasso coefficients.
        
    """

    lasso = Lasso(alpha=alpha)
    lasso.fit(X, y)
    
    # PROTEÇÃO CRÍTICA
    limit = min(len(names), len(lasso.coef_))
    
    return [names[i] for i in range(limit) if lasso.coef_[i] != 0]

In [137]:
def consensus_features(X, y, names, min_votes=2, rfe_n_features=None):
    """
    Feature selection via:
    1) Lasso + Ridge voting to define a candidate pool
    2) RFE (with Ridge) applied only to that pool

    Args:
        X (ndarray): Scaled training features
        y (array): Training target
        names (list of str): Feature names (aligned with X)
        min_votes (int): Minimum votes from Lasso + Ridge to enter the pool
        rfe_n_features (int or None): Final number of features selected by RFE.
                                      If None, keeps all pooled features.
        ridge_alpha (float): Ridge alpha
        lasso_alpha (float): Lasso alpha

    Returns:
        selected_features (list of str)
    """

    # --- 1) Free voting phase (no cardinality constraints) ---
    f_lasso = select_lasso(X, y, names)
    f_ridge = select_ridge(X, y, names)

    votes = Counter(f_lasso + f_ridge)

    pool = [feat for feat, count in votes.items() if count >= min_votes]

    # Safety net: if pool collapses, fall back to union
    if len(pool) == 0:
        pool = sorted(set(f_lasso) | set(f_ridge))

    # --- 2) RFE refinement phase ---
    if rfe_n_features is None or rfe_n_features >= len(pool):
        return sorted(pool)

    # Reduce X to pooled features
    pool_idx = [names.index(f) for f in pool]
    X_pool = X[:, pool_idx]

    refined = select_rfe(
        X_pool,
        y,
        pool,
        n_features=rfe_n_features
    )

    return sorted(refined)

In [26]:
def best_model_per_metric(metrics_df):
    best = {}

    best["R2"] = metrics_df["R2"].idxmax()
    best["MAE"] = metrics_df["MAE"].idxmin()
    best["MSE"] = metrics_df["MSE"].idxmin()
    best["MAPE"] = metrics_df["MAPE"].idxmin()
    best["MedAE"] = metrics_df["MedAE"].idxmin()

    return best

## Models

To identify the model that best fits our data, several regression algorithms with different biases were trained and validated. This comparative approach allows us to evaluate both linear and non-linear relationships, as well as parametric and ensemble-based methods.

The following seven models were considered:
- Linear Regression
- Ridge Regression
- Random Forest
- Decision Tree
- Gradient Boosting
- MLP (Multi-layer Perceptron)
- LinearSVM (Support Vector Machine): since SVM doesn't scale well for datasets with more than 10k rows

In [71]:
models = {
    #'Linear Regression': LinearRegression(),
    #'Ridge': Ridge(alpha=1),
    'Random Forest': RandomForestRegressor(n_estimators= 100, random_state=42, max_depth = 20, n_jobs=-1),
    #'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300,learning_rate=0.1,max_depth=5,random_state=42),
    #'Neural Network (MLP)': MLPRegressor(hidden_layer_sizes=(100,50), max_iter=500, random_state=42),
    #'LinearSVR': LinearSVR()- Para este temos ou de mudar para StandardScaler ou ver melhor os outliers
}

## Metrics

To be able to compare the performance of the trained models, 5 evaluation metrics were used. Each metric capture either a degree of explainability of the model or the error on the predicted and observed values.

The following metrics were considered:
- **R²**: Measures the proportion of the variance for a dependent variable. Provides an indication of how well the data points fit a statistical model

- **MAE** (Mean Absolute Error): Computes the average absolute difference between predicted and actual values. (Tells by how much the model is missing)

- **MSE** (Mean Squared Error): Calculates the average of the squared prediction errors. By squaring the errors, this metric penalizes large mistakes more heavily than MAE.

- **MAPE** (Mean Absolute Percentage Error): Measures the average absolute error as a percentage of the true values. 

- **MedAE** (Median Absolute Error): Computes the median of the absolute prediction errors. 

Using multiple metrics avoids over-optimizing for a single criterion and helps reveal trade-offs between accuracy, robustness, and sensitivity to extreme errors.

In [11]:
metrics = {
    'R2': r2_score,
    'MAE': mean_absolute_error,
    'MSE': mean_squared_error,
    'MAPE': mean_absolute_percentage_error,
    'MedAE': median_absolute_error
}

## Cross-Validation

To train the models we opted by the Cross-Validation strategy. This method compares the models by dividing the training data into 5. From those 5 datasets 1 will be the validation, that will be changed into a training dataset in the next interation, making every "mini" dataset be the validation dataset once.

In [72]:
#Using the Standard Cross-Validation to evaluate the model
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [73]:
results = {name: {m: [] for m in metrics} for name in models}

To ensure that the CV (Cross-Validation) doesn't leak any data, we need to make a few steps before spliting and going straith to training. We decided to divide this task into 5 steps:

#### Data Spliting

To start the CV, we create a loop that divides the dataset into training and validation. using this loop, we can guarantee that every mini dataset is used for validation once.

#### Preprocessing

All preprocessing steps are fitted exclusively on the training data and then applied to the validation data. This prevents data leakage and ensures a fair evaluation. In this step, we take care of:

- **Missing values** in numerical features are imputed using the median computed from the training fold.

- **Categorical features** are encoded using one-hot encoding.

- Unknown categories in the validation set are safely ignored.

After encoding, numerical and categorical features are concatenated into a single feature matrix.

#### Feature Scaling

All features are scaled using Min-Max normalization, fitted only on the training data and then applied to the validation data.

We need to do this step because we use scale-sensitive models such as Support Vector Machines (LinearSVR) and Neural Networks (MLP).

#### Feature Selection

Feature selection is performed inside each cross-validation fold, using only the training data.

A consensus-based feature selection strategy is applied, combining:

- Recursive Feature Elimination (RFE),

- Ridge regression coefficient filtering,

- Lasso regression sparsity.

Only features selected by a majority of these methods are retained. The same subset of features is then applied to both training and validation sets for that fold.

This approach favors stable and robust features while avoiding information leakage.

#### Model Training and Evaluation

Each model is trained on the selected features of the training fold and evaluated on the corresponding validation fold.

Predictions are assessed using the evaluation metrics already inicialized a few cells above

The results for each model and metric are stored across all folds, enabling later aggregation and comparison.

In [138]:
fold = 1
for train_index, val_index in cv.split(X,y):
    print(f"--> A processar Fold {fold}/5...")
# 1- Data Split
    X_train_fold = X.iloc[train_index].copy()
    X_val_fold = X.iloc[val_index].copy()

    y_train_fold = y[train_index]
    y_val_fold = y[val_index]
    

# 2- Preprocessing
    # Seperate the numeric columns
    numeric_cols = X_train_fold.select_dtypes(include=np.number).columns

    # Impute the null values
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train_fold[numeric_cols])  

    # Apply the transformation
    X_train_fold[numeric_cols] = imputer.transform(X_train_fold[numeric_cols])
    X_val_fold[numeric_cols] = imputer.transform(X_val_fold[numeric_cols])

    #Encoding
    categorical_cols = X_train_fold.select_dtypes(include=['object', 'bool']).columns

    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_fold[categorical_cols])

    X_train_encoded = encoder.transform(X_train_fold[categorical_cols])
    X_val_encoded = encoder.transform(X_val_fold[categorical_cols])

    X_train_final = np.hstack([X_train_fold[numeric_cols].values, X_train_encoded])
    X_val_final = np.hstack([X_val_fold[numeric_cols].values, X_val_encoded])

    encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
    all_feature_names = list(numeric_cols) + list(encoded_feature_names)

# 3- Scaling
    scaler = MinMaxScaler()
    scaler.fit(X_train_final) # Fit on TRAIN
    
    X_train_scaled = scaler.transform(X_train_final)
    X_val_scaled = scaler.transform(X_val_final)


# 4 - Feature Selection
    # Chama a função de consenso passando os nomes das colunas
    selected = consensus_features(X_train_scaled, y_train_fold, all_feature_names)
    print(selected)
    print(len(selected))
    # Encontrar os índices das colunas selecionadas
    sel_idx = [all_feature_names.index(f) for f in selected]

    # Reduzir as matrizes de treino e validação
    X_train_final = X_train_scaled[:, sel_idx]
    X_val_final   = X_val_scaled[:, sel_idx]
    
# 5 - Training and Evaluation
    for name, model in models.items():
        m = clone(model)
        m.fit(X_train_final, y_train_fold)
        pred = m.predict(X_val_final)
        
        for m_name, func in metrics.items():
            score = func(y_val_fold, pred)
            results[name][m_name].append(score)
            
    fold += 1



--> A processar Fold 1/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


['Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Unknown', 'Brand_VW', 'age', 'engineSize', 'fuelType_Hybrid', 'fuelType_Petrol', 'mileage_per_year', 'model_2 Series', 'model_3 Series', 'model_4 Series', 'model_A Class', 'model_A1', 'model_Aygo', 'model_B Class', 'model_C Class', 'model_Corsa', 'model_Fabia', 'model_Fiesta', 'model_Focus', 'model_GLA Class', 'model_GLC Class', 'model_GLE Class', 'model_Golf', 'model_Grandland X', 'model_Ka', 'model_Kodiaq', 'model_Octavia', 'model_Polo', 'model_Q3', 'model_Q5', 'model_Q7', 'model_R8', 'model_S Class', 'model_Tiguan', 'model_Up', 'model_X3', 'model_X5', 'model_X7', 'model_Yaris', 'model_i10', 'mpg_engineSize', 'transmission_Manual', 'transmission_Semi-Auto', 'transmission_Unknown']
50
--> A processar Fold 2/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


['Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Unknown', 'Brand_VW', 'age', 'engineSize', 'fuelType_Hybrid', 'fuelType_Petrol', 'fuelType_Unknown', 'mileage_per_year', 'model_2 Series', 'model_3 Series', 'model_4 Series', 'model_A Class', 'model_A1', 'model_Aygo', 'model_B Class', 'model_C Class', 'model_Caravelle', 'model_Corsa', 'model_Fabia', 'model_Fiesta', 'model_Focus', 'model_GLA Class', 'model_GLC Class', 'model_GLE Class', 'model_Golf', 'model_Grandland X', 'model_Ka', 'model_Kodiaq', 'model_Octavia', 'model_Polo', 'model_Q3', 'model_Q5', 'model_Q7', 'model_R8', 'model_S Class', 'model_Tiguan', 'model_Up', 'model_X3', 'model_X5', 'model_X7', 'model_Yaris', 'model_i10', 'mpg', 'transmission_Manual', 'transmission_Semi-Auto', 'transmission_Unknown']
52
--> A processar Fold 3/5...


...

['Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Unknown', 'Brand_VW', 'age', 'engineSize', 'fuelType_Hybrid', 'fuelType_Petrol', 'mileage_per_year', 'model_2 Series', 'model_3 Series', 'model_4 Series', 'model_A1', 'model_Aygo', 'model_B Class', 'model_C Class', 'model_Corsa', 'model_Fabia', 'model_Fiesta', 'model_Focus', 'model_GLA Class', 'model_GLC Class', 'model_GLE Class', 'model_Golf', 'model_Grandland X', 'model_Ka', 'model_Kodiaq', 'model_Octavia', 'model_Polo', 'model_Q3', 'model_Q5', 'model_Q7', 'model_Q8', 'model_S Class', 'model_Tiguan', 'model_Up', 'model_X3', 'model_X5', 'model_X7', 'model_Yaris', 'model_i10', 'mpg', 'transmission_Manual', 'transmission_Semi-Auto', 'transmission_Unknown']
49
--> A processar Fold 4/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


['Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Unknown', 'Brand_VW', 'age', 'engineSize', 'fuelType_Hybrid', 'fuelType_Petrol', 'mileage_per_year', 'model_2 Series', 'model_3 Series', 'model_4 Series', 'model_A Class', 'model_A1', 'model_Aygo', 'model_B Class', 'model_C Class', 'model_Caravelle', 'model_Corsa', 'model_Fabia', 'model_Fiesta', 'model_Focus', 'model_GLA Class', 'model_GLC Class', 'model_GLE Class', 'model_Golf', 'model_Grandland X', 'model_Ka', 'model_Kodiaq', 'model_Octavia', 'model_Polo', 'model_Q3', 'model_Q5', 'model_Q7', 'model_Q8', 'model_S Class', 'model_Tiguan', 'model_Up', 'model_X3', 'model_X5', 'model_X7', 'model_Yaris', 'model_i10', 'mpg', 'transmission_Manual', 'transmission_Semi-Auto', 'transmission_Unknown']
51
--> A processar Fold 5/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


['Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Unknown', 'Brand_VW', 'age', 'engineSize', 'fuelType_Hybrid', 'fuelType_Petrol', 'mileage_per_year', 'model_2 Series', 'model_3 Series', 'model_4 Series', 'model_A Class', 'model_A1', 'model_Aygo', 'model_B Class', 'model_C Class', 'model_Corsa', 'model_Fabia', 'model_Fiesta', 'model_Focus', 'model_GLA Class', 'model_GLC Class', 'model_GLE Class', 'model_Golf', 'model_Grandland X', 'model_Ka', 'model_Kodiaq', 'model_Octavia', 'model_Polo', 'model_Q3', 'model_Q5', 'model_Q7', 'model_S Class', 'model_Tiguan', 'model_Up', 'model_X3', 'model_X5', 'model_X7', 'model_Yaris', 'model_i10', 'mpg', 'transmission_Manual', 'transmission_Semi-Auto', 'transmission_Unknown']
49


## Models Score

After completing CV, we need to see, which model got the better results across all the folds. To do that, we created a Dataframe that will show the **mean** result for each model

We decided to sort the results by **MAE**, since its the metric that is evaluating in the Kaggle competition.

In [139]:
# --- RESULTS TABLE ---
# Convert the dictionary of lists into a DataFrame
results_df = pd.DataFrame(results).T

# Calculate the mean of the 5 folds for each metric
# (This gives you the final "Score" for each model)
metrics_table = results_df.map(lambda x: np.mean(x))

# Display the table sorted by R2 (or MAE)
print("\n--- Model Performance (Cross-Validation Mean) ---")
display(metrics_table.round(4).sort_values(by='MAE', ascending=True))


--- Model Performance (Cross-Validation Mean) ---


,R2,MAE,MSE,MAPE,MedAE
Random Forest,0.9171,1602.2383,7.861710e+06,0.0996,961.0793
Gradient Boosting,0.9177,1724.4530,7.806693e+06,0.1080,1126.7870


## Best Model Decider

In [130]:
best_models = best_model_per_metric(metrics_table)

for metric, model in best_models.items():
    print(f"{metric}: {model}")


R2: Gradient Boosting
MAE: Random Forest
MSE: Gradient Boosting
MAPE: Random Forest
MedAE: Random Forest


Since *Random Forest* and *Gradient Boosting* were the only two models that got the best result in the CV. Besides Random Forest winning in the most important metrics: *MAPE* and *MedAE* (our opinion), we will emsemble both models, to see if we can get even better results.

## Hyperparameter tunning

In this section, we will tune both algorithms, trying to find the best combination of parameters possible. To do so, we will start with the Random Forest and then Gradient Boosting.

For both algorithms we will use GridSearch. We inicialize a dictionary for each algorithm and fill them with a range of parameters.

#### Random Forest

For Random Forest we have:
- **n_estimators**: Interval for the number of trees
- **max_depth**: Interval for the maximum depth that a tree can have (otherwise it overfits)
- **max_features**: Number of features when splitting
- **min_samples_split**: Minimum number of samples required to split a node
- **min_samples_leaf**: Minimum number of samples required to be at a leaf node
- **bootstrap**: Uses samples to build each tree

In [33]:
# --- GRID SEARCH "DEEP DIVE" PARA RANDOM FOREST ---
# Total de Combinações: ~300 (x 5 Folds = 1500 treinos)
# Estimativa de Tempo: 20 a 40 minutos (dependendo do CPU)

rf_grid = {
    'model_name': ['Random Forest'],
    'model': [RandomForestRegressor(random_state=42, n_jobs=-1)],
    'params': {
        # 1. Quantidade de Árvores:
        # Testamos valores mais altos para estabilizar a previsão
        'n_estimators': [500, 800],

        # 2. Profundidade da Árvore:
        # None = cresce até ao fim (pode overfitar). 20 e 30 controlam.
        'max_depth': [20, 30],

        # 3. Robustez (Features):
        # 'sqrt' vê menos colunas (mais rápido, menos overfit).
        # 1.0 vê todas as colunas (melhor se tiveres poucas features boas).
        'max_features': ['sqrt', 1.0],

        # 4. Controlo de Folhas (O segredo do MAE):
        # min_samples_split: Mínimo de carros para dividir um nó (2, 5, 10)
        # min_samples_leaf: Mínimo de carros numa folha final (1, 2, 4)
        # Valores mais altos aqui evitam que o modelo decore carros únicos.
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],

        # 5. Bootstrap:
        # True = Usa amostragem aleatória (padrão, reduz variância).
        # False = Usa o dataset todo para cada árvore (pode ser brutalmente preciso mas perigoso).
        'bootstrap': [True]
    }
}

grids_to_test = [rf_grid]
rf_grid_results = {}


In [34]:
print("A iniciar Grid Search Manual...")

# 2. LOOP DE VALIDAÇÃO (O mesmo de sempre)
fold = 1
for train_index, val_index in cv.split(X, y):
    print(f"--> A processar Fold {fold}/5...")
    
    # --- A) DATA SPLIT ---
    X_train_fold = X.iloc[train_index].copy()
    X_val_fold = X.iloc[val_index].copy()
    y_train_fold = y[train_index]
    y_val_fold = y[val_index]

    # --- B) PREPROCESSING ---
    # 1. Impute
    numeric_cols = X_train_fold.select_dtypes(include=np.number).columns
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train_fold[numeric_cols])
    X_train_fold[numeric_cols] = imputer.transform(X_train_fold[numeric_cols])
    X_val_fold[numeric_cols] = imputer.transform(X_val_fold[numeric_cols])

    # 2. Encode
    categorical_cols = X_train_fold.select_dtypes(include=['object', 'bool']).columns
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_fold[categorical_cols])
    X_train_encoded = encoder.transform(X_train_fold[categorical_cols])
    X_val_encoded = encoder.transform(X_val_fold[categorical_cols])

    X_train_processed = np.hstack([X_train_fold[numeric_cols].values, X_train_encoded])
    X_val_processed = np.hstack([X_val_fold[numeric_cols].values, X_val_encoded])

    # 3. Scale
    scaler = MinMaxScaler()
    scaler.fit(X_train_processed)
    X_train_scaled = scaler.transform(X_train_processed)
    X_val_scaled = scaler.transform(X_val_processed)

    # 4. Feature Selection (SelectFromModel - RF)
    # Chama a função de consenso passando os nomes das colunas
    selected = consensus_features(X_train_scaled, y_train_fold, all_feature_names)

    # Encontrar os índices das colunas selecionadas
    sel_idx = [all_feature_names.index(f) for f in selected]

    # Reduzir as matrizes de treino e validação
    X_train_final = X_train_scaled[:, sel_idx]
    X_val_final   = X_val_scaled[:, sel_idx]
    
    # --- C) GRID SEARCH INTERNO ---
    for entry in grids_to_test:
        base_model = entry['model'][0]
        base_name = entry['model_name'][0]
        
        # CORREÇÃO AQUI: Usar ParameterGrid para gerar as combinações
        param_grid = list(ParameterGrid(entry['params']))
        
        for params in param_grid:
            # Criar nome único para esta configuração
            combo_name = f"{base_name} | {str(params)}"
            
            # Inicializar se for novo
            if combo_name not in rf_grid_results:
                rf_grid_results[combo_name] = {m: [] for m in metrics}
            
            # Clonar e configurar
            m = clone(base_model)
            m.set_params(**params)
            
            m.fit(X_train_final, y_train_fold)
            pred = m.predict(X_val_final)
            
            # Guardar Scores
            for m_name, func in metrics.items():
                score = func(y_val_fold, pred)
                rf_grid_results[combo_name][m_name].append(score)
    
    fold += 1

A iniciar Grid Search Manual...
--> A processar Fold 1/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.896e+11, tolerance: 5.647e+08
  model = cd_fast.enet_coordinate_descent(


--> A processar Fold 2/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.132e+11, tolerance: 5.650e+08
  model = cd_fast.enet_coordinate_descent(


--> A processar Fold 3/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.025e+11, tolerance: 5.623e+08
  model = cd_fast.enet_coordinate_descent(


--> A processar Fold 4/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.063e+11, tolerance: 5.601e+08
  model = cd_fast.enet_coordinate_descent(


KeyboardInterrupt: 

In [ ]:
# MOSTRAR O VENCEDOR
print("\n--- Tuning Concluído! ---")
rf_results_df = pd.DataFrame(rf_grid_results).T
rf_final_metrics = rf_results_df.map(lambda x: np.mean(x))

print("\n--- TOP 5 Melhores Configurações (Ordenado por MAE) ---")
display(rf_final_metrics.round(4).sort_values(by='MAE', ascending=True).head(5))


--- Tuning Concluído! ---

--- TOP 5 Melhores Configurações (Ordenado por MAE) ---


,R2,MAE,MSE,MAPE,MedAE
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 800}",0.8758,2076.1136,1.178435e+07,0.1279,1230.4408
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 500}",0.8758,2076.7625,1.178667e+07,0.1279,1230.6424
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 800}",0.8751,2078.5529,1.185422e+07,0.1280,1231.8251
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 500}",0.8751,2079.0198,1.185422e+07,0.1280,1232.2865
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 800}",0.8748,2082.5798,1.188090e+07,0.1282,1232.9731


#### Gradient Boosting

For Gradient Boosting we have:
- **n_estimators**: The number of boosting stages to perform (Since its robust to overfit we used a large number)
- **learning_rate**: Shrinks the contribution of each tree by the values in the parameter
- **max_depth**: Limits the number of nodes in the tree
- **subsample**: Fraction of samples to be used for fitting the base learners
- **max_features**: Number of features to consider

In [ ]:
# --- GRID SEARCH: GRADIENT BOOSTING (O "Challenger") ---
# O GB é sequencial, por isso demora mais que o RF a treinar.
# Esta grelha foca-se nos parâmetros essenciais para bater o MAE.

gb_grid = {
    'model_name': ['Gradient Boosting'],
    'model': [GradientBoostingRegressor(random_state=42)],
    'params': {
        # 1. Potência (Árvores vs. Velocidade de Aprendizagem)
        # Regra: Se baixares o learning_rate, tens de subir os n_estimators.
        # 0.05 com 500/800 árvores costuma ser muito preciso.
        'n_estimators': [500, 800],
        'learning_rate': [0.05, 0.1],

        # 2. Estrutura da Árvore (Devem ser mais pequenas que no RF)
        # O GB usa árvores "fracas" (weak learners). Profundidade 3 a 5 é o padrão.
        # Testamos 7 para ver se ele precisa de capturar interações mais complexas.
        'max_depth': [3, 5, 7],

        # 3. Robustez (Stochastic Gradient Boosting)
        # subsample < 1.0 faz com que ele use apenas uma fração dos dados para cada árvore.
        # Isto reduz o overfitting e muitas vezes melhora o resultado final.
        'subsample': [0.8],

        # 4. Features por Árvore
        'max_features': ['sqrt', 1.0]
    }
}

# Se quiseres correr APENAS o Gradient Boosting (recomendado para poupar tempo):
grids_to_test = [gb_grid]
gb_grid_results = {}
# Se quiseres correr RF e GB ao mesmo tempo (vai demorar muito):
# grids_to_test = [rf_grid, gb_grid]

In [ ]:
print("A iniciar Grid Search Manual para o Gradient Boosting...")

# 2. LOOP DE VALIDAÇÃO (O mesmo de sempre)
fold = 1
for train_index, val_index in cv.split(X, y):
    print(f"--> A processar Fold {fold}/5...")
    
    # --- A) DATA SPLIT ---
    X_train_fold = X.iloc[train_index].copy()
    X_val_fold = X.iloc[val_index].copy()
    y_train_fold = y[train_index]
    y_val_fold = y[val_index]

    # --- B) PREPROCESSING ---
    # 1. Impute
    numeric_cols = X_train_fold.select_dtypes(include=np.number).columns
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train_fold[numeric_cols])
    X_train_fold[numeric_cols] = imputer.transform(X_train_fold[numeric_cols])
    X_val_fold[numeric_cols] = imputer.transform(X_val_fold[numeric_cols])

    # 2. Encode
    categorical_cols = X_train_fold.select_dtypes(include=['object', 'bool']).columns
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_fold[categorical_cols])
    X_train_encoded = encoder.transform(X_train_fold[categorical_cols])
    X_val_encoded = encoder.transform(X_val_fold[categorical_cols])

    X_train_processed = np.hstack([X_train_fold[numeric_cols].values, X_train_encoded])
    X_val_processed = np.hstack([X_val_fold[numeric_cols].values, X_val_encoded])

    # 3. Scale
    scaler = MinMaxScaler()
    scaler.fit(X_train_processed)
    X_train_scaled = scaler.transform(X_train_processed)
    X_val_scaled = scaler.transform(X_val_processed)

    # 4. Feature Selection (SelectFromModel - RF)
    # Chama a função de consenso passando os nomes das colunas
    selected = consensus_features(X_train_scaled, y_train_fold, all_feature_names)

    # Encontrar os índices das colunas selecionadas
    sel_idx = [all_feature_names.index(f) for f in selected]

    # Reduzir as matrizes de treino e validação
    X_train_final = X_train_scaled[:, sel_idx]
    X_val_final   = X_val_scaled[:, sel_idx]
    
    # --- C) GRID SEARCH INTERNO ---
    for entry in grids_to_test:
        base_model = entry['model'][0]
        base_name = entry['model_name'][0]
        
        # CORREÇÃO AQUI: Usar ParameterGrid para gerar as combinações
        param_grid = list(ParameterGrid(entry['params']))
        
        for params in param_grid:
            # Criar nome único para esta configuração
            combo_name = f"{base_name} | {str(params)}"
            
            # Inicializar se for novo
            if combo_name not in gb_grid_results:
                gb_grid_results[combo_name] = {m: [] for m in metrics}
            
            # Clonar e configurar
            m = clone(base_model)
            m.set_params(**params)
            
            m.fit(X_train_final, y_train_fold)
            pred = m.predict(X_val_final)
            
            # Guardar Scores
            for m_name, func in metrics.items():
                score = func(y_val_fold, pred)
                gb_grid_results[combo_name][m_name].append(score)
    
    fold += 1


A iniciar Grid Search Manual para o Gradient Boosting...
--> A processar Fold 1/5...


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.896e+11, tolerance: 5.647e+08
  model = cd_fast.enet_coordinate_descent(


KeyboardInterrupt: 

In [ ]:
# MOSTRAR O VENCEDOR
print("\n--- Tuning Concluído! ---")
gb_results_df = pd.DataFrame(gb_grid_results).T
gb_final_metrics = gb_results_df.map(lambda x: np.mean(x))

print("\n--- TOP 5 Melhores Configurações (Ordenado por MAE) ---")
display(gb_final_metrics.round(4).sort_values(by='MAE', ascending=True).head(5))

## Best Model

In [91]:
# Célula [54] - Best Model (Mantida como está)
final_model = RandomForestRegressor(
    n_estimators=800, 
    max_depth=20, 
    max_features=1.0,  
    min_samples_split=5,
    min_samples_leaf=1,
    bootstrap=True,
    random_state=42
)

In [92]:
# --- FASE 3: FULL DEPLOYMENT (SUBMISSÃO FINAL) ---
final_model = GradientBoostingRegressor(
    n_estimators=800,
    learning_rate=0.1,
    max_depth=7,
    max_features=1.0,
    subsample=0.8,
    random_state=42
)

In [140]:
# 2. Carregar e Preparar Dados (Pipeline de Transformação e Feature Selection)

# Carregar o teste original 
X_test_kaggle = pd.read_csv('../new_datasets/X_test_preprocessed.csv')
X_full = X.copy()
y_full = y.copy()

# --- A) Impute ---
numeric_cols = X_full.select_dtypes(include=np.number).columns
imputer = SimpleImputer(strategy='median')
imputer.fit(X_full[numeric_cols])

X_full[numeric_cols] = imputer.transform(X_full[numeric_cols])
X_test_kaggle[numeric_cols] = imputer.transform(X_test_kaggle[numeric_cols])

# --- B) Encode ---
categorical_cols = X_full.select_dtypes(include=['object', 'bool']).columns
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoder.fit(X_full[categorical_cols])

X_full_enc = encoder.transform(X_full[categorical_cols])
X_test_enc = encoder.transform(X_test_kaggle[categorical_cols])

# Juntar Arrays e obter nomes de todas as features (numéricas + dummy)
X_full_processed = np.hstack([X_full[numeric_cols].values, X_full_enc])
X_test_processed = np.hstack([X_test_kaggle[numeric_cols].values, X_test_enc])
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
all_feature_names = list(numeric_cols) + list(encoded_feature_names) # Essencial para o consenso

# --- C) Scale ---
scaler = MinMaxScaler()
scaler.fit(X_full_processed)

X_full_scaled = scaler.transform(X_full_processed)
X_test_scaled = scaler.transform(X_test_processed)

# --- D) Feature Selection (Aplicação do consenso na TOTALIDADE dos dados de treino) ---
print("A realizar Feature Selection por consenso...")
selected = consensus_features(X_full_scaled, y_full, all_feature_names)

# Encontrar os índices das colunas selecionadas nos arrays escalados
sel_idx = [all_feature_names.index(f) for f in selected]

# Reduzir as matrizes de treino e teste
X_full_final = X_full_scaled[:, sel_idx]
X_test_final = X_test_scaled[:, sel_idx]

print(f"Dados prontos! Features finais: {X_full_final.shape[1]}")
print(f"Features selecionadas: {selected}")

A realizar Feature Selection por consenso...
Dados prontos! Features finais: 49
Features selecionadas: ['Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Unknown', 'Brand_VW', 'age', 'engineSize', 'fuelType_Hybrid', 'fuelType_Petrol', 'mileage_per_year', 'model_2 Series', 'model_3 Series', 'model_4 Series', 'model_A Class', 'model_A1', 'model_Aygo', 'model_B Class', 'model_C Class', 'model_Corsa', 'model_Fabia', 'model_Fiesta', 'model_Focus', 'model_GLA Class', 'model_GLC Class', 'model_GLE Class', 'model_Golf', 'model_Grandland X', 'model_Ka', 'model_Kodiaq', 'model_Octavia', 'model_Polo', 'model_Q3', 'model_Q5', 'model_Q7', 'model_S Class', 'model_Tiguan', 'model_Up', 'model_X3', 'model_X5', 'model_X7', 'model_Yaris', 'model_i10', 'mpg', 'transmission_Manual', 'transmission_Semi-Auto', 'transmission_Unknown']


In [141]:
# --- VALIDAÇÃO DO ENSEMBLE (CROSS-VALIDATION) ---
# Objetivo: Confirmar se misturar GB (70%) + RF (30%) bate os modelos individuais.

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold

print("=== A INICIAR VALIDAÇÃO DO ENSEMBLE (5 FOLDS) ===")

# 1. Configurar Validação
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
r2_scores = []
fold = 1

# 2. Loop de Validação (O teu pipeline padrão)
for train_index, val_index in kf.split(X, y):
    # A) Data Split
    X_train_fold, X_val_fold = X.iloc[train_index].copy(), X.iloc[val_index].copy()
    y_train_fold, y_val_fold = y[train_index], y[val_index]
    
    # B) Preprocessing (Impute -> Encode -> Scale)
    # Limpar infinitos
    num_cols = X_train_fold.select_dtypes(include=np.number).columns
    X_train_fold[num_cols] = X_train_fold[num_cols].replace([np.inf, -np.inf], np.nan)
    X_val_fold[num_cols] = X_val_fold[num_cols].replace([np.inf, -np.inf], np.nan)
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train_fold[num_cols])
    X_train_fold[num_cols] = imputer.transform(X_train_fold[num_cols])
    X_val_fold[num_cols] = imputer.transform(X_val_fold[num_cols])
    
    # Encode
    cat_cols = X_train_fold.select_dtypes(include=['object', 'bool']).columns
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_fold[cat_cols])
    
    X_train_enc = encoder.transform(X_train_fold[cat_cols])
    X_val_enc = encoder.transform(X_val_fold[cat_cols])
    
    # Juntar e Escalar
    X_train_proc = np.hstack([X_train_fold[num_cols].values, X_train_enc])
    X_val_proc = np.hstack([X_val_fold[num_cols].values, X_val_enc])
    
    scaler = MinMaxScaler()
    scaler.fit(X_train_proc)
    X_train_scaled = scaler.transform(X_train_proc)
    X_val_scaled = scaler.transform(X_val_proc)
    
    # Feature Selection (Consenso)
    feat_names = list(num_cols) + list(encoder.get_feature_names_out(cat_cols))
    
    try:
        # Usa a função se existir, senão usa todas
        selected = consensus_features(X_train_scaled, y_train_fold, feat_names)
        sel_idx = [feat_names.index(f) for f in selected]
        X_train_final = X_train_scaled[:, sel_idx]
        X_val_final = X_val_scaled[:, sel_idx]
    except:
        X_train_final = X_train_scaled
        X_val_final = X_val_scaled

    # C) TREINO DOS 2 MODELOS
    # Gradient Boosting (O Campeão)
    gb = GradientBoostingRegressor(n_estimators=800, learning_rate=0.1, max_depth=7, 
                                   max_features=1.0, subsample=0.8, random_state=42)
    gb.fit(X_train_final, y_train_fold)
    
    # Random Forest (O Estável)
    rf = RandomForestRegressor(n_estimators=800, max_depth=20, max_features=1.0, 
                               min_samples_leaf=1, min_samples_split=5, n_jobs=-1, random_state=42)
    rf.fit(X_train_final, y_train_fold)
    
    # D) PREVISÃO E MISTURA (ENSEMBLE)
    pred_gb = gb.predict(X_val_final)
    pred_rf = rf.predict(X_val_final)
    
    # AQUI TESTAMOS A PROPORÇÃO 70/30
    pred_ensemble = (0.7 * pred_gb) + (0.3 * pred_rf)
    
    # Métricas
    mae = mean_absolute_error(y_val_fold, pred_ensemble)
    r2 = r2_score(y_val_fold, pred_ensemble)
    
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print(f"Fold {fold}: MAE Ensemble = {mae:.2f} (GB puro seria: {mean_absolute_error(y_val_fold, pred_gb):.2f})")
    fold += 1

# RESULTADO FINAL
print("\n--- RESULTADOS FINAIS DO ENSEMBLE (70/30) ---")
print(f"MAE Médio: {np.mean(mae_scores):.4f}")
print(f"R2 Médio:  {np.mean(r2_scores):.4f}")

if np.mean(mae_scores) < 1479:
    print("✅ DECISÃO: O Ensemble MELHORA o resultado! Avança para o deployment.")
else:
    print("⚠️ DECISÃO: O Ensemble NÃO melhora. Submete apenas o Gradient Boosting.")

=== A INICIAR VALIDAÇÃO DO ENSEMBLE (5 FOLDS) ===


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Fold 1: MAE Ensemble = 1355.17 (GB puro seria: 1370.22)


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Fold 2: MAE Ensemble = 1333.12 (GB puro seria: 1347.28)


...

Fold 3: MAE Ensemble = 1364.95 (GB puro seria: 1385.52)


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Fold 4: MAE Ensemble = 1370.28 (GB puro seria: 1381.38)


c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Fold 5: MAE Ensemble = 1365.26 (GB puro seria: 1385.75)

--- RESULTADOS FINAIS DO ENSEMBLE (70/30) ---
MAE Médio: 1357.7566
R2 Médio:  0.9415
✅ DECISÃO: O Ensemble MELHORA o resultado! Avança para o deployment.


## Final Ensemble and Kaggle Submission

In [142]:
print("A treinar Ensemble Final (GB + RF)...")

# 1. Definir Modelos (Melhores Parâmetros)
gb_model = GradientBoostingRegressor(
    n_estimators=800, learning_rate=0.1, max_depth=7, 
    max_features=1.0, subsample=0.8, random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=800, max_depth=20, max_features=1.0, 
    min_samples_leaf=1, min_samples_split=5, n_jobs=-1, random_state=42
)

# 2. Treinar
print("--> Treinando Gradient Boosting...")
gb_model.fit(X_full_final, y_full)

print("--> Treinando Random Forest...")
rf_model.fit(X_full_final, y_full)

# 3. Prever
print("--> Gerando previsões...")
pred_gb = gb_model.predict(X_test_final)
pred_rf = rf_model.predict(X_test_final)

# 4. Misturar (70% GB + 30% RF)
final_preds = (0.7 * pred_gb) + (0.3 * pred_rf)

# 5. Guardar CSV
submission = pd.DataFrame({
    'CarID': pd.read_csv('../original_datasets/test.csv')['carID'],
    'price': final_preds
})
submission.to_csv('kaggle_submission_ensemble.csv', index=False)
print("✅ SUCESSO! 'kaggle_submission_ensemble.csv' criado.")

A treinar Ensemble Final (GB + RF)...
--> Treinando Gradient Boosting...
--> Treinando Random Forest...
--> Gerando previsões...
✅ SUCESSO! 'kaggle_submission_ensemble.csv' criado.
